In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load all CSV files in the current folder
data_dir = Path(".")
csv_files = list(data_dir.glob("*.csv"))

print("Files found:", len(csv_files))
for f in csv_files:
    print(f.name)

# Combine all CSVs
data_list = [pd.read_csv(f) for f in csv_files]
data = pd.concat(data_list, ignore_index=True)

print("Original shape:", data.shape)
data.head()

# Clean column names
data.columns = data.columns.str.strip()

# Remove duplicate columns, if any
data = data.loc[:, ~data.columns.duplicated()]

print("Shape after fixing duplicate columns:", data.shape)

# Check labels
print(data["Label"].value_counts())

# Drop duplicate rows
print("Duplicate rows:", data.duplicated().sum())
data.drop_duplicates(inplace=True)

print("Shape after dropping duplicate rows:", data.shape)

# Replace infinity with NaN
data.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Missing values:")
print(data.isna().sum()[data.isna().sum() > 0])

# Fill missing values with median
for col in data.columns:
    if data[col].isna().sum() > 0 and col != "Label":
        data[col] = data[col].fillna(data[col].median())

print("Total missing values:", data.isna().sum().sum())

# Convert labels to binary
data["Label"] = data["Label"].apply(lambda x: 0 if x == "BENIGN" else 1)

print(data["Label"].value_counts())

# Drop constant columns
constant_cols = data.columns[data.nunique() <= 1]
print("Constant columns:", list(constant_cols))

data.drop(columns=constant_cols, inplace=True)

print("Final cleaned shape:", data.shape)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = data.drop("Label", axis=1)
y = data["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train:", X_train_scaled.shape)
print("X_test:", X_test_scaled.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True))
print("y_test distribution:")
print(y_test.value_counts(normalize=True))



Files found: 8
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Original shape: (2830743, 80)
Shape after fixing duplicate columns: (2830743, 79)
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
